# Variante GRU — Santiago Diaz

Entrenamiento del `CharRNN(cell_type='gru')` para generar nombres de dinosaurios.

Reutiliza el núcleo compartido en [src/](../src/): `dataset.py`, `model.py`, `train.py`, `sample.py`.

**Hiperparámetros**: `embed_dim=32`, `hidden_dim=128`, `num_layers=1`, `lr=1e-3`, `batch_size=64`.

Ejecuta este notebook desde la **raíz del repo** (no desde `notebooks/`) para que los imports relativos funcionen.

In [ ]:
import sys, os
from pathlib import Path

REPO_ROOT = Path.cwd()
while REPO_ROOT.name and not (REPO_ROOT / 'CLAUDE.md').exists():
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print('cwd:', os.getcwd())

In [ ]:
import torch
import matplotlib.pyplot as plt
from Parte_1_Generador_Caracteres.src.dataset import make_dataloaders, load_names
from Parte_1_Generador_Caracteres.src.model import CharRNN
from Parte_1_Generador_Caracteres.src.sample import sample_one

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## 1. Datos

In [ ]:
names = load_names('data/dinos.csv')
print(f'total nombres: {len(names)}')
print(f'longitud min/max: {min(map(len, names))}/{max(map(len, names))}')
print('ejemplos:', names[:5])

In [ ]:
train_loader, val_loader, vocab, max_len = make_dataloaders('data/dinos.csv', batch_size=64)
print(f'vocab_size={len(vocab)}, max_len={max_len}')
print(f'train batches={len(train_loader)}, val batches={len(val_loader)}')

## 2. Entrenamiento

Llamamos directamente a `train.train(args)` para reutilizar el bucle del módulo.

In [ ]:
from argparse import Namespace
from Parte_1_Generador_Caracteres.src.train import train

args = Namespace(
    data='data/dinos.csv',
    cell='gru',
    epochs=50,
    batch_size=64,
    embed=32,
    hidden=128,
    layers=1,
    lr=1e-3,
    patience=5,
    checkpoint='Parte_1_Generador_Caracteres/models/gru_SantiagoDiaz.pt',
    run_tag='SantiagoDiaz',
)
history, best_val = train(args)
print(f'best val loss: {best_val:.4f}')

## 3. Curvas de aprendizaje

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history['train_loss'], label='train')
ax.plot(history['val_loss'], label='val')
ax.set_xlabel('Epoch'); ax.set_ylabel('CrossEntropy')
ax.set_title('GRU — Santiago Diaz')
ax.legend(); ax.grid(alpha=0.3)
fig.savefig('reports/learning_curves_gru.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Muestreo rápido (sanity check)

Generamos 15 nombres con `temperature=1.0, top_p=0.9` para validar visualmente la calidad antes del barrido completo (notebook `03_sampling.ipynb`).

In [ ]:
from Parte_1_Generador_Caracteres.src.sample import load_model, generate_unique

model, vocab, max_len = load_model(args.checkpoint, device=device)
seen = set(load_names('data/dinos.csv'))
names_gen = generate_unique(model, vocab, max_len, n=15,
                            temperature=1.0, top_k=None, top_p=0.9,
                            seen=seen, device=device)
for name in names_gen:
    print(' -', name)

## 5. Promoción a `best_model.pt` (si gana)

Comparar `best_val` con el de Alan (RNN) y Juan Camilo (LSTM). Si esta GRU es la mejor, promover el checkpoint a `best_model.pt`:

```python
import shutil
shutil.copy(args.checkpoint, 'Parte_1_Generador_Caracteres/models/best_model.pt')
```

**Coordinar con el equipo antes de ejecutar** — el checkpoint final lo consume el sampling (Parte 1) y el backend Lambda (Parte 4).